# Week 1 Lab: Know Your Data

## Lab Setup

In [1]:
# setup.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("UCI_Credit_Card.csv")
print(df.shape)                                           # (30000, 25)
print(df["default.payment.next.month"].mean().round(3))   # ~0.221 default rate

(30000, 25)
0.221


## Section 1: First contact

### Exercise 1.1: Shape, types, and a column that isn’t there

In [2]:
# section1_profile.py
print(df.shape)
print(df.dtypes.value_counts())
print(df.isna().sum().sum())        # total blanks in the whole file

(30000, 25)
float64    13
int64      12
Name: count, dtype: int64
0


In [3]:
print([c for c in df.columns if c.startswith("PAY_") and "AMT" not in c])

['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']


Question 1.1

Answer: I'd rather be wrong about PAY_0 being mis-named because if it's actually a missing month and we've renamed it, we could've accidentally introduced misalignment.

To check this we can use ```print(df[["PAY_0", "BILL_AMT1", "PAY_AMT1"]].head(10))``` and cross-reference the BILL_AMT and PAY_AMT columns.


### Exercise 1.2: Audit the data against its documentation

In [4]:
# section1_audit.py
for c in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(f"{c:10s} {sorted(df[c].unique().tolist())}")
print(f"{'PAY_0':10s} {sorted(df['PAY_0'].unique().tolist())}")

SEX        [1, 2]
EDUCATION  [0, 1, 2, 3, 4, 5, 6]
MARRIAGE   [0, 1, 2, 3]
PAY_0      [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]


In [5]:
print("EDUCATION in {0,5,6}:", df.EDUCATION.isin([0, 5, 6]).sum())
print("MARRIAGE == 0      :", (df.MARRIAGE == 0).sum())
print("PAY_0 in {-2, 0}   :", df.PAY_0.isin([-2, 0]).sum(),
      f"({100 * df.PAY_0.isin([-2, 0]).mean():.1f}%)")

EDUCATION in {0,5,6}: 345
MARRIAGE == 0      : 54
PAY_0 in {-2, 0}   : 17496 (58.3%)


Question 1.2

Answer: Our options are: 1 - Drop the rows (58.3% of the column is affected, so we'd lose the majority of the data)
                         2 - Treat them as a new category (e.g. "no delay / account settled")
                         3 - Impute with the closest documented code (e.g. map -2 and 0 to -1 "paid duly")

We can treat -2 and 0 as a distinct category, e.g. "no delay", and keep them in the model while documenting our assumptions.

In [6]:
rate = df.groupby("PAY_0")["default.payment.next.month"].agg(["mean", "size"])
print(rate.assign(mean=rate["mean"].round(3)).to_string())

        mean   size
PAY_0              
-2     0.132   2759
-1     0.168   5686
 0     0.128  14737
 1     0.339   3688
 2     0.691   2667
 3     0.758    322
 4     0.684     76
 5     0.500     26
 6     0.545     11
 7     0.778      9
 8     0.579     19


Task 1.2

Answer: -2 and 0 most likely mean early repayment or no credit used that month, since their default rates sit below even "paid duly". That said, inferring meaning from default rates alone is weak evidence, a codebook tells you what a code is, behaviour only tells you what it does, and those aren't the same thing.


### Checkpoint 1

1. How many rows and columns, and how many literal blanks, does this file have?

Answer: The file has 30,000 rows and 25 columns, with zero literal blanks.

2. Which three columns contain codes that aren't in the documentation, and how many rows does each affect?

Answer: EDUCATION - 345 rows (codes 0, 5, 6)
                MARRIAGE - 54 rows (code 0)
                PAY_0 - 17,496 rows (codes -2 and 0)

3. Why is the PAY_0 finding more serious than the EDUCATION one, even though both are "undocumented codes"?

Answer: Because 58.3% of PAY_0 is undocumented, and PAY_0 is the single most predictive feature in the dataset. In EDUCATION, 345 undocumented rows out of 30,000 is negligible. 